In [1]:
# 03_model_training.ipynb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import os,sys

project_root = os.path.abspath("..")   # ← 根據你的資料夾結構調整
sys.path.insert(0, project_root)

print("Current working dir:", os.getcwd())
print("Project root added:", project_root)

Current working dir: /Users/mankin/Desktop/Anomaly_Detection_on_Graph/notebooks
Project root added: /Users/mankin/Desktop/Anomaly_Detection_on_Graph


In [2]:
# 確保每次訓練前都從頭開始處理資料，避免舊的 processed 檔案影響結果
processed_path = "../data/processed/elliptic_processed.pt"
if os.path.exists(processed_path):
    os.remove(processed_path)
    print("已成功刪除舊的 processed 檔案！下次執行 EllipticDataset 會重新處理")
else:
    print("找不到舊檔案，已經是乾淨狀態")

已成功刪除舊的 processed 檔案！下次執行 EllipticDataset 會重新處理


In [3]:
# ====================== 設定 Random Seed（確保實驗可重現） ======================

from src.utils import set_seed
set_seed(42)

 Random seed 已固定為 42，所有隨機性已關閉，實驗結果可完全重現。


In [4]:
# ====================== 引入 config ======================

from src.config import Config
cfg = Config()               

In [5]:
#====================== override 你想改的參數 ======================
    
cfg.exp_name = "exp_009_final_tuning"

cfg.hidden_dim = 128
cfg.num_layers = 3
cfg.lr = 0.0015                    # 再降低一點 lr
cfg.epochs = 400                   # 可以跑更久
cfg.patience = 30                  # 再提高 patience
cfg.use_degree = True
cfg.use_pagerank = True
cfg.aggregator = "mean"
cfg.dropout = 0.5

In [6]:
# ====================== Create Experiment Folder ======================

from src.utils import create_experiment

# Create experiment directory and save config.yaml
exp_dir = create_experiment(
    cfg=cfg,
    description=""   # 實驗描述（可選）e.g. GraphSAGE with degree + PageRank features, temporal split, mean aggregator
)

print(f"實驗目錄建立完成：{exp_dir}")

本次實驗名稱：exp_009_final_tuning
實驗資料夾：../experiments/exp_009_final_tuning
config.yaml 已自動建立
實驗設定已儲存至: ../experiments/exp_009_final_tuning/config.yaml
實驗目錄建立完成：../experiments/exp_009_final_tuning


In [7]:
# ====================== 載入資料 ======================

# start before initialize dataset, to ensure processed file is deleted
from src.data import EllipticDataset   # 從 src.data 模組中引入 EllipticDataset 類別，這是我們之前定義的資料集類別，用來處理 Elliptic 資料集。

dataset = EllipticDataset(
    root=str(cfg.data_dir), 
    use_degree=cfg.use_degree,
    use_pagerank=cfg.use_pagerank
)
data = dataset[0]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x = data.x.to(device)
edge_index = data.edge_index.to(device)
y = data.y.to(device)

print(f"資料載入完成：{x.shape[0]} nodes, {edge_index.shape[1]} edges, {x.shape[1]} features")

/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing...


開始處理 Elliptic 資料集...
MARKER: process() method called - About to read classes
Nodes: 203769, Features dim: 165
Sample txIds (col 0): [230425980, 5530458, 232022460, 232438397, 230460314]
Sample time steps (col 1): [1, 1, 1, 1, 1]
edgelist 有效行數: 234355
Sample edges txId1 (first 5): [230425980, 232022460, 230460314, 230333930, 232013274]
MARKER: Starting edge matching loop
有效 directed edges: 234355
匹配成功率: 100.00% (234355/234355)
無向 edges: 468710
加入 degree → x dim: 166
正在計算 PageRank (networkx), 請耐心等待...
加入 PageRank → x dim: 167


/Users/mankin/Desktop/Anomaly_Detection_on_Graph/src/data.py:173: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  torch.save((self.data, self.slices), self.processed_paths[0])


處理完成！
最終: 203769 nodes, 468710 edges
資料載入完成：203769 nodes, 468710 edges, 167 features


Done!
/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([DataEdgeAttr])` to allowlist this global.
  out = fs.torch_load(path)


In [8]:
# ====================== Initialize Model ======================

from src.models import ImprovedGraphSAGE

model = ImprovedGraphSAGE(
    in_channels=x.shape[1],           # e.g. 167 (features + degree + pagerank)
    hidden_dim=cfg.hidden_dim,        
    num_layers=cfg.num_layers,        
    dropout=cfg.dropout,              
    aggregator=cfg.aggregator         
).to(device)

print(model)

 Kaiming initialization + Residual Connections applied (layers=3, hidden=128)
ImprovedGraphSAGE(in=167, hidden=128, layers=3, agg=mean, dropout=0.5, init=Kaiming+Residual)


In [9]:
# ====================== Data Splitting ======================

from src.split import split_data

# Perform data split (temporal split is recommended)
train_idx, val_idx, test_idx = split_data(
    data=data,
    y=y,
    device=device,
    temporal_split=True,          # Set to False if you want random split
    test_time_threshold=34,
    val_size=0.2,
    random_state=42
)

使用 Temporal Split（推薦）
訓練: 23915 | 驗證: 5979 | 測試: 16670


In [10]:
# ====================== Training + Evaluation + Saving ======================

from src.train import train_model

best_val_auc, best_model_path, test_auc, test_auprc, best_epoch = train_model(
    model=model,
    x=x,
    edge_index=edge_index,
    y=y,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,           # ← important
    cfg=cfg,
    device=device,
    exp_dir=exp_dir,             # ← This will automatically save results.json + model_best.pt
    save_best=True
)

print(f"\n Training finished!")
print(f"Test AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f}")

 Using Weighted CrossEntropy Loss with weights: [1.0, 12.0]
Starting training for 400 epochs (patience=30) with Focal Loss...

Epoch   0 | Loss: 1.3085 | Val AUC: 0.7506
Epoch   5 | Loss: 0.7771 | Val AUC: 0.9040
Epoch  10 | Loss: 0.5837 | Val AUC: 0.9215
Epoch  15 | Loss: 0.4751 | Val AUC: 0.9279
Epoch  20 | Loss: 0.4046 | Val AUC: 0.9321
Epoch  25 | Loss: 0.3710 | Val AUC: 0.9374
Epoch  30 | Loss: 0.3647 | Val AUC: 0.9430
Epoch  35 | Loss: 0.3373 | Val AUC: 0.9483
Epoch  40 | Loss: 0.3212 | Val AUC: 0.9533
Epoch  45 | Loss: 0.3109 | Val AUC: 0.9577
Epoch  50 | Loss: 0.2960 | Val AUC: 0.9619
Epoch  55 | Loss: 0.2862 | Val AUC: 0.9657
Epoch  60 | Loss: 0.2714 | Val AUC: 0.9690
Epoch  65 | Loss: 0.2589 | Val AUC: 0.9720
Epoch  70 | Loss: 0.2528 | Val AUC: 0.9747
Epoch  75 | Loss: 0.2403 | Val AUC: 0.9768
Epoch  80 | Loss: 0.2214 | Val AUC: 0.9785
Epoch  85 | Loss: 0.2157 | Val AUC: 0.9800
Epoch  90 | Loss: 0.2089 | Val AUC: 0.9812
Epoch  95 | Loss: 0.1982 | Val AUC: 0.9823
Epoch 100 | L